$\textbf{SRM Uncertainty Propagation Koaxial Example}$

In [2]:
#DEFINITIONS
import skrf as rf
from skrf.calibration import TRL, TUGMultilineTRL
import numpy as np
import matplotlib.pyplot as plt
from u_srm import uSRM
import metas_unclib as munc
from aux_functions import extract_switch_terms, extract_table, interpolate_to_freq, s2t, t2s, inv_propagate_all_contributors
from aux_functions import cov_of_2x2,cov_of_scalar,inv_propagate_one_port,cov_of_scalar,forward_one_port, forward_two_port
munc.use_linprop()

Auxiliary Functions

$\textbf{Add Test Data}$

In [3]:

short_p1 = rf.Network('CoaxMeas_data/Short_P1.s1p')['0-40GHz']
short_p2 = rf.Network('CoaxMeas_data/Short_P2.s1p')['0-40GHz']
open_p1 = rf.Network('CoaxMeas_data/Open_P1.s1p')['0-40GHz']
open_p2 = rf.Network('CoaxMeas_data/Open_P2.s1p')['0-40GHz']
load_p1 = rf.Network('CoaxMeas_data/Match_P1.s1p')['0-40GHz']
load_p2 = rf.Network('CoaxMeas_data/Match_P2.s1p')['0-40GHz']
reciprocal = rf.Network('CoaxMeas_data/Network.s2p')['0-40GHz']
recipocal_short_p1= rf.Network('CoaxMeas_data/Network_Short_P1.s1p')['0-40GHz']
recipocal_open_p1= rf.Network('CoaxMeas_data/Network_Open_P1.s1p')['0-40GHz']
recipocal_load_p1= rf.Network('CoaxMeas_data/Network_Match_P1.s1p')['0-40GHz']
FWST,RWST =  extract_switch_terms('Thru_with_switch_terms.tex')
attenuator = rf.Network('CoaxMeas_data/Attenuator.s2p')['0-40GHz']
mismatch_p1 = rf.Network('CoaxMeas_data/Missmatch_P1.s1p')['0-40GHz']
mismatch_p2 = rf.Network('CoaxMeas_data/Missmatch_P2.s1p')['0-40GHz']
attenuator_solr = rf.Network('METAS_SOLR_De_Emb/Attenuator.s2p')['0-40GHz']
freq = short_p1.frequency


$\textbf{Construct Ideals}$

In [4]:

reciprocal_est = np.zeros((len(freq),2,2),dtype=complex)
reciprocal_est[0:len(freq),0,0] = 0*np.ones((len(freq)),dtype=complex)
reciprocal_est[0:len(freq),0,1] = np.ones((len(freq)),dtype=complex)
reciprocal_est[0:len(freq),1,0] = np.ones((len(freq)),dtype=complex)
reciprocal_est[0:len(freq),1,1] = 0*np.ones((len(freq)),dtype=complex)
reciprocal_est = rf.Network(frequency=freq,s=reciprocal_est,z0=50)

reciprocal_est = rf.Network('METAS_SOLR_De_Emb/Network.s2p')['0-40GHz']

short_est = -1*np.ones(len(freq))
open_est = 1*np.ones(len(freq))
short_est = rf.Network('Reference_Standards/Short_F.s1p')['0-40GHz'].s[:,0,0]
open_est = rf.Network('Reference_Standards/Open_F.s1p')['0-40GHz'].s[:,0,0]
#match_est = 0*np.ones((len(freq)))
match_est = rf.Network('Reference_Standards/Match_F.s1p')['0-40GHz'].s[:,0,0]
match_def = rf.Network('Reference_Standards/Match_F.s1p')['0-40GHz']

match_est_2p = np.zeros((len(freq),2,2),dtype=complex)
match_est_2p[0:len(freq),0,0] = match_est
match_est_2p[0:len(freq),1,1] = match_est
match_est_2p = rf.Network(frequency=freq,s=match_est_2p,z0=50)

short_est_2p = np.zeros((len(freq),2,2),dtype=complex)
short_est_2p[0:len(freq),0,0] = short_est
short_est_2p[0:len(freq),1,1] = short_est
short_est_2p = rf.Network(frequency=freq,s=short_est_2p,z0=50)

open_est_2p = np.zeros((len(freq),2,2),dtype=complex)
open_est_2p[0:len(freq),0,0] = open_est
open_est_2p[0:len(freq),1,1] = open_est
open_est_2p = rf.Network(frequency=freq,s=open_est_2p,z0=50)

$\textbf{Import Unceratinty Contributors}$

In [5]:
noise_data = extract_table("Uncertainty_Contributor_Data/Noise_Floor.tex")
drift_data = extract_table("Uncertainty_Contributor_Data/Drift.tex")
cable_data = extract_table("Uncertainty_Contributor_Data/Cable.tex")
connector_data = extract_table("Uncertainty_Contributor_Data/Connector.tex")
linearity_data = extract_table("Uncertainty_Contributor_Data/Linearity.tex")
open_data = extract_table("Uncertainty_Contributor_Data/Open_F.tex")
short_data = extract_table("Uncertainty_Contributor_Data/Short_F.tex")
load_data = extract_table("Uncertainty_Contributor_Data/Match_F.tex")

# The following assumtions and conversions are made:
# Port 1 and Port 2 are the same, so just take one column for each contributor
# Conversion form dB to linear scale is done by 10^(dB/20) for magnitude wrapped around 0 
# and 10^(dB/20)-1 for magnitude wrapped around 1 (e.g. linearity or tracking drift)

nf_i  = interpolate_to_freq(10**(noise_data[:, 1]/20),      noise_data[:, 0], freq)**2
nt_i  = interpolate_to_freq(10**(noise_data[:, 5]/20) - 1,  noise_data[:, 0], freq)**2
l_i   = interpolate_to_freq(10**(linearity_data[:, 9]/20) - 1, linearity_data[:, 0], freq)**2

dd_i = interpolate_to_freq(10**(drift_data[:, 2]/20),        drift_data[:, 0], freq)**2
dt_i = interpolate_to_freq(10**(drift_data[:, 3]/20)-1,      drift_data[:, 0], freq)**2
ds_i = interpolate_to_freq(10**(drift_data[:, 5]/20)-1,        drift_data[:, 0], freq)**2
dm_i = interpolate_to_freq(10**(drift_data[:, 7]/20),        drift_data[:, 0], freq)**2

cr_i = interpolate_to_freq(10**(cable_data[:, 1]/20),        cable_data[:, 0], freq)**2
ct_i = interpolate_to_freq(10**(cable_data[:, 2]/20)-1,        cable_data[:, 0], freq)**2

rr_i = interpolate_to_freq(10**(connector_data[:, 1]/20),     connector_data[:, 0], freq)**2


# The standards are saved in lin magnitude! and frequ. in MHz
ss_i = interpolate_to_freq(short_data[:, 2],        short_data[:, 0]*1e6, freq)**2
so_i = interpolate_to_freq(open_data[:, 2],      open_data[:, 0]*1e6, freq)**2
sm_i = interpolate_to_freq(load_data[:, 2],        load_data[:, 0]*1e6, freq)**2



Create Drift UNC Martices

In [6]:
DD = []
DT = []
DM = []
DS = []
D = np.zeros((len(freq),2,2),dtype=munc.ucomplex)
for i in range(len(freq)):
    DD.append(munc.ucomplex(0,covariance=[[dd_i[i]/np.sqrt(2),0],[0,dd_i[i]/np.sqrt(2)]]))  #1/sqrt(2) factor is needed to "convert" from mag to real and imaginary parts
    DT.append(munc.ucomplex(1,covariance=[[dt_i[i]/np.sqrt(2),0],[0,dt_i[i]/np.sqrt(2)]]))
    DS.append(munc.ucomplex(1,covariance=[[ds_i[i]/np.sqrt(2),0],[0,ds_i[i]/np.sqrt(2)]]))
    DM.append(munc.ucomplex(0,covariance=[[dm_i[i]/np.sqrt(2),0],[0,dm_i[i]/np.sqrt(2)]]))
    D[i] = np.array([[DD[i],DT[i]/DS[i]],[DS[i],DM[i]]])
#D = np.array([[DD,DT/DS],[DS,DM]])

#print(D)

Create Noise UNC Martices

In [7]:
NF = []
NT = []
L = []

N_L = np.zeros((len(freq),2,2),dtype=munc.ucomplex)
for i in range(len(freq)):
    NF.append(munc.ucomplex(0,covariance=[[nf_i[i]/np.sqrt(2),0],[0,nf_i[i]/np.sqrt(2)]]))
    NT.append(munc.ucomplex(1,covariance=[[nt_i[i]/np.sqrt(2),0],[0,nt_i[i]/np.sqrt(2)]]))
    L.append(munc.ucomplex(1,covariance=[[l_i[i]/np.sqrt(2),0],[0,l_i[i]/np.sqrt(2)]]))
    N_L[i] = np.array([[NF[i],NT[i]*L[i]],[1,0]])

#print(N_L)

Create Cable Matrix

In [8]:
CR = []
CT = []
C = np.zeros((len(freq),2,2),dtype=munc.ucomplex)
for i in range(len(freq)):
    CR.append(munc.ucomplex(0,covariance=[[cr_i[i]/np.sqrt(2),0],[0,cr_i[i]/np.sqrt(2)]]))
    CT.append(munc.ucomplex(1,covariance=[[ct_i[i]/np.sqrt(2),0],[0,ct_i[i]/np.sqrt(2)]]))
    C[i] = np.array([[CR[i],CT[i]],[CT[i],CR[i]]])

#print(C)

Create Match Cov Matrix

In [9]:
unc_match_def = np.zeros((len(freq),2,2),dtype=complex)

for i in range(len(freq)):
    unc_match_def[i] = np.array([[sm_i[i]/np.sqrt(2),0],[0,sm_i[i]/np.sqrt(2)]])
    


Create Connector Matrix

In [10]:
RR = []
R = np.zeros((len(freq),2,2),dtype=munc.ucomplex)
for i in range(len(freq)):
    RR.append(munc.ucomplex(0,covariance=[[rr_i[i]/np.sqrt(2),0],[0,rr_i[i]/np.sqrt(2)]]))

    R[i] = np.array([[RR[i],1],[1,RR[i]]])
    

In [11]:
u_short = 0
u_open = 0
u_match = 0
u_reciprocal = 0
u_match_def = 0
u_switch_term = 0
u_reciprocal_GammaA_short = 0
u_reciprocal_GammaA_open = 0
u_reciprocal_GammaA_load = 0

In [12]:
calSRM_no_unc = uSRM(symmetric=[short_p1,short_p2, open_p1, open_p2, load_p1, load_p2],
            est_symmetric=[short_est_2p,  open_est_2p, match_est_2p],
            reciprocal=reciprocal,est_reciprocal=reciprocal_est,
            reciprocal_GammaA = [recipocal_short_p1, recipocal_open_p1, recipocal_load_p1],
            matchA=load_p1, matchB=load_p2,
            matchA_def=match_def, matchB_def=match_def,
            switch_term_fw=FWST, switch_term_bw=RWST,
            usymmetric=[u_short,u_short,u_open,u_open,u_match,u_match],
            ureciprocal=u_reciprocal,
            ureciprocal_GammaA=[u_reciprocal_GammaA_short, u_reciprocal_GammaA_open, u_reciprocal_GammaA_load],
            umatchA_def=u_match_def, umatchB_def=u_match_def,
            umatchA=u_match, umatchB=u_match,
            usw_fw=u_switch_term, usw_bw=u_switch_term)

calSRM_no_unc.run()
test = calSRM_no_unc.apply_cal(attenuator)


SRM is running...


Extract Error Boxes from Cal (without unc yet)

In [13]:
k = calSRM_no_unc.k
a = calSRM_no_unc.A
b = calSRM_no_unc.B

$\textbf{Costruct Cov-Matrices for each Measurement; Combine it for forward and inverse unc.-propagation.}$

$\textbf{Assign What Measurement got which UNC Contributor}$\
\
Short_P1: Re-Connection and Small-Movement -> $U_{SP1}=U_{N+L} + I_{1-p}(U_D + U_C + U_R)$\
Short_P2: Re-Connection and Small-Movement -> $U_{SP2}=U_{N+L} + I_{1-p}(U_D + U_C + U_R )$\
Open_P1: Re-Connection and Small-Movement -> $U_{OP1}=U_{N+L} + I_{1-p}(U_D + U_C + U_R )$\
Open_P2: Re-Connection and Small-Movement -> $U_{OP2}=U_{N+L} + I_{1-p}(U_D + U_C + U_R )$\
Match_P1: Re-Connection and Small-Movement -> $U_{MP1}=U_{N+L} + I_{1-p}(U_D + U_C + U_R )$\
Match_P2: Re-Connection and Small-Movement -> $U_{MP2}=U_{N+L} + I_{1-p}(U_D + U_C + U_R )$\
Network_Short_P1: Re-Connection and Small-Movement -> $U_{NSP1}=U_{N+L} + I_{1-p}(U_D + U_C + U_R )$\
Network_Open_P1: Re-Connection and Small-Movement -> $U_{NOP1}=U_{N+L} + I_{1-p}(U_D + U_C + U_R )$\
Network_Match_P1: Re-Connection and Small-Movement -> $U_{NMP1}=U_{N+L} + I_{1-p}(U_D + U_C + U_R )$\
Mismatch_P1: Re-Connection and Small-Movement -> $U_{MMP1}=U_{N+L} + I_{1-p}(U_D + U_C + U_R )$\
Mismatch_P2: Re-Connection and Small-Movement -> $U_{MMP2}=U_{N+L} + I_{1-p}(U_D + U_C + U_R )$\
Network: Re-Connection and Small-Movement -> $U_{N}=U_{N+L} + I_{2-p}(U_D + U_C + U_R )$\
Attenuator: Re-Connection and Small-Movement -> $U_{A}=U_{N+L} + I_{2-p}(U_D + U_C + U_R )$


In [14]:
#Short P1 and P2

short_est = -1*np.ones(len(freq))
unc_short_p1 = np.zeros((len(freq),2,2),dtype=complex)
unc_short_p2 = np.zeros((len(freq),2,2),dtype=complex)

for i in range(len(freq)):
    g   = inv_propagate_one_port([D[i], C[i], R[i]],munc.ucomplex(short_est[i]),a[i], port="A")
    g_tot = forward_one_port([N_L[i]], g, port="A")
    unc_short_p1[i] = cov_of_scalar(g_tot)
    
    d = inv_propagate_one_port([D[i], C[i], R[i]],munc.ucomplex(short_est[i]),b[i], port="B")
    d_tot = forward_one_port([N_L[i]], d, port="B")
    unc_short_p2[i] = cov_of_scalar(d_tot)
    


In [15]:
#Open P1 and P2


open_est = 1*np.ones(len(freq))
unc_open_p1 = np.zeros((len(freq),2,2),dtype=complex)
unc_open_p2 = np.zeros((len(freq),2,2),dtype=complex)
for i in range(len(freq)):
    g   = inv_propagate_one_port([D[i], C[i], R[i]],munc.ucomplex(open_est[i]),a[i], port="A")
    g_tot = forward_one_port([N_L[i]], g, port="A")
    unc_open_p1[i] = cov_of_scalar(g_tot)
        
    d = inv_propagate_one_port([D[i], C[i], R[i]],munc.ucomplex(open_est[i]),b[i], port="B")
    d_tot = forward_one_port([N_L[i]], d, port="B")
    unc_open_p2[i] = cov_of_scalar(d_tot)
    

KeyboardInterrupt: 

In [ ]:
#Match P1 and P2


#match_est = 0*np.ones((len(freq)))
match_est = rf.Network('Reference_Standards/Match_F.s1p')['0-40GHz'].s[:,0,0]
unc_match_p1 = np.zeros((len(freq),2,2),dtype=complex)
unc_match_p2 = np.zeros((len(freq),2,2),dtype=complex)
for i in range(len(freq)):
    g   = inv_propagate_one_port([D[i], C[i], R[i]],munc.ucomplex(match_est[i]),a[i], port="A")
    g_tot = forward_one_port([N_L[i]], g, port="A")
    unc_match_p1[i] = cov_of_scalar(g_tot)
    
    d = inv_propagate_one_port([D[i], C[i], R[i]],munc.ucomplex(match_est[i]),b[i], port="B")
    d_tot = forward_one_port([N_L[i]], d, port="B")
    unc_match_p2[i] = cov_of_scalar(d_tot)


In [ ]:
# Network Short, same as Short P1, just here to be changed if network would be very lossy

short_est = -1*np.ones(len(freq))
unc_network_short_p1 = np.zeros((len(freq),2,2),dtype=complex)


for i in range(len(freq)):
    g   = inv_propagate_one_port([D[i], C[i], R[i]],munc.ucomplex(short_est[i]),a[i], port="A")
    g_tot = forward_one_port([N_L[i]], g, port="A")
    unc_network_short_p1[i] = cov_of_scalar(g_tot)

In [ ]:
# Network Open, same as Open P1, just here to be changed if network would be very lossy

open_est = 1*np.ones(len(freq))
unc_network_open_p1 = np.zeros((len(freq),2,2),dtype=complex)

for i in range(len(freq)):
    g   = inv_propagate_one_port([D[i], C[i], R[i]],munc.ucomplex(open_est[i]),a[i], port="A")
    g_tot = forward_one_port([N_L[i]], g, port="A")
    unc_network_open_p1[i] = cov_of_scalar(g_tot)

In [ ]:
# Network Match, same as Open P1, just here to be changed if network would be very reflective


#match_est = 0*np.ones((len(freq)))
match_est = rf.Network('Reference_Standards/Match_F.s1p')['0-40GHz'].s[:,0,0]
unc_network_match_p1 = np.zeros((len(freq),2,2),dtype=complex)

for i in range(len(freq)):
    g   = inv_propagate_one_port([D[i], C[i], R[i]],munc.ucomplex(match_est[i]),a[i], port="A")
    g_tot = forward_one_port([N_L[i]], g, port="A")
    unc_network_match_p1[i] = cov_of_scalar(g_tot)

In [ ]:
# Network 

P = np.array([[0,1],[1,0]])

# Inverse propagation of the contributors for each frequency point 2-Port for reciprocal Standard
unc_network = np.zeros((len(freq),8,8),dtype=complex)
for i in range(len(freq)):
    Da, Ca, Ra = D[i], C[i], R[i]               # Port-A
    Db, Cb, Rb = D[i], C[i], R[i]               # Port-B
    Db = P @ Db @ P                     # Port-B- is flipped
    m = inv_propagate_all_contributors([Da, Ca, Ra], [Rb, Cb, Db],a[i], b[i], k[i],mid_S=None)
    m_tot = forward_two_port([N_L[i]],m)
    unc_network[i] = cov_of_2x2(m_tot)
    

$\textbf{Perform SRM with Uncertainties}$

In [ ]:
calSRM = uSRM(symmetric=[short_p1,short_p2, open_p1, open_p2, load_p1, load_p2],
            est_symmetric=[short_est_2p,  open_est_2p, match_est_2p],
            reciprocal=reciprocal,est_reciprocal=reciprocal_est,
            reciprocal_GammaA = [recipocal_short_p1, recipocal_open_p1, recipocal_load_p1],
            matchA=load_p1, matchB=load_p2,
            matchA_def=match_def, matchB_def=match_def,
            switch_term_fw=FWST.s11, switch_term_bw=RWST.s11,
            usymmetric=[unc_short_p1,unc_short_p2, unc_open_p1, unc_open_p2, unc_match_p1, unc_match_p2],
            ureciprocal=unc_network,
            ureciprocal_GammaA=[unc_network_short_p1, unc_network_open_p1, unc_network_match_p1],
            umatchA_def=unc_match_def, umatchB_def=unc_match_def,
            umatchA=unc_match_p1, umatchB=unc_match_p2,
            usw_fw=u_switch_term, usw_bw=u_switch_term)

In [ ]:
calSRM.u_run()


uSRM is running...
Frequency: 0.10 GHz ... DONE!


C:\Program Files (x86)\METAS\Metas.UncLib\metas_unclib.py:198: ComplexWarning: Casting complex values to real discards the imaginary part
  c = [_Number(float(bi)) for bi in b.flatten('F')]


Frequency: 0.14 GHz ... DONE!
Frequency: 0.19 GHz ... DONE!
Frequency: 0.23 GHz ... DONE!
Frequency: 0.27 GHz ... DONE!
Frequency: 0.31 GHz ... DONE!
Frequency: 0.36 GHz ... DONE!
Frequency: 0.40 GHz ... DONE!
Frequency: 0.44 GHz ... DONE!
Frequency: 0.49 GHz ... DONE!
Frequency: 0.53 GHz ... DONE!
Frequency: 0.57 GHz ... DONE!
Frequency: 0.61 GHz ... DONE!
Frequency: 0.66 GHz ... DONE!
Frequency: 0.70 GHz ... DONE!
Frequency: 0.74 GHz ... DONE!
Frequency: 0.79 GHz ... DONE!
Frequency: 0.83 GHz ... DONE!
Frequency: 0.87 GHz ... DONE!
Frequency: 0.92 GHz ... DONE!
Frequency: 0.96 GHz ... DONE!
Frequency: 1.00 GHz ... DONE!
Frequency: 1.04 GHz ... DONE!
Frequency: 1.09 GHz ... DONE!
Frequency: 1.13 GHz ... DONE!
Frequency: 1.17 GHz ... DONE!
Frequency: 1.22 GHz ... DONE!
Frequency: 1.26 GHz ... DONE!
Frequency: 1.30 GHz ... DONE!
Frequency: 1.34 GHz ... DONE!
Frequency: 1.39 GHz ... DONE!
Frequency: 1.43 GHz ... DONE!
Frequency: 1.47 GHz ... DONE!
Frequency: 1.52 GHz ... DONE!
Frequency:

In [ ]:
ref_solr = extract_table("Attenuator.tex")

In [ ]:
DUT1,unc = calSRM.u_apply_cal(attenuator)

plt.figure()
plt.title('S21')
plt.plot(freq.f, 20*np.log10(np.abs(DUT1.to_dataframe('s')['s 21'])),label='uSRM',color='green')
plt.fill_between(freq.f, (20*np.log10(np.abs(DUT1.to_dataframe('s')['s 21'])-(np.abs(unc[:,1,0])))), (20*np.log10(np.abs(DUT1.to_dataframe('s')['s 21'])+(np.abs(unc[:,1,0])))), color='green', alpha=0.5, label='uSRM uncertainty')
plt.plot(freq.f, 20*np.log10(np.abs(ref_solr[:, 5]+1j*ref_solr[:, 7])),label='Attenuator (METAS SOLR)',color='orange')
plt.fill_between(freq.f, (20*np.log10(np.abs(ref_solr[:, 5]+1j*ref_solr[:, 7])-(np.abs(ref_solr[:, 6]+1j*ref_solr[:, 8])))), (20*np.log10(np.abs(ref_solr[:, 5]+1j*ref_solr[:, 7])+(np.abs(ref_solr[:, 6]+1j*ref_solr[:, 8])))), color='orange', alpha=0.5, label='Attenuator uncertainty')
plt.grid()
plt.legend()

plt.figure()
plt.title('S11')
plt.plot(freq.f, 20*np.log10(np.abs(DUT1.to_dataframe('s')['s 11'])),label='uSRM',color='green')
plt.fill_between(freq.f, (20*np.log10(np.abs(DUT1.to_dataframe('s')['s 11'])-(np.abs(unc[:,0,0])))), (20*np.log10(np.abs(DUT1.to_dataframe('s')['s 11'])+(np.abs(unc[:,0,0])))), color='green', alpha=0.5, label='uSRM uncertainty')
plt.plot(freq.f, 20*np.log10(np.abs(ref_solr[:, 1]+1j*ref_solr[:, 3])),label='Attenuator (METAS SOLR)',color='orange')
plt.fill_between(freq.f, (20*np.log10(np.abs(ref_solr[:, 1]+1j*ref_solr[:, 3])-(np.abs(ref_solr[:, 2]+1j*ref_solr[:, 4])))), (20*np.log10(np.abs(ref_solr[:, 1]+1j*ref_solr[:, 3])+(np.abs(ref_solr[:, 2]+1j*ref_solr[:, 4])))), color='orange', alpha=0.5, label='Attenuator uncertainty')
plt.grid()
plt.legend()

plt.show()

NullReferenceException: Der Objektverweis wurde nicht auf eine Objektinstanz festgelegt.
   bei Metas.UncLib.Core.Unc.UncList`1.IsDiagonalMatrix(Number[][] covariance) in D:\svn_work\217.01 S-Parameter\Software\Metas.UncLib\source\Metas.UncLib.Core\Core.Unc\UncList.cs:Zeile 215.
   bei Metas.UncLib.Core.Unc.UncList`1.InitMultivariateNormal(Number[] values, Number[][] covariance, InputId id, String desc) in D:\svn_work\217.01 S-Parameter\Software\Metas.UncLib\source\Metas.UncLib.Core\Core.Unc\UncList.cs:Zeile 124.
   bei Metas.UncLib.Core.Unc.UncList`1.InitMultivariateT(Number[] values, Number[][] covariance, Double idof, InputId id, String desc) in D:\svn_work\217.01 S-Parameter\Software\Metas.UncLib\source\Metas.UncLib.Core\Core.Unc\UncList.cs:Zeile 198.
   bei Metas.UncLib.LinProp.UncList.InitMultivariateT(Number[] values, Number[][] covariance, Double idof, InputId id, String desc) in D:\svn_work\217.01 S-Parameter\Software\Metas.UncLib\source\Metas.UncLib.LinProp\LinProp\UncLis